- Load the data from the source_table 
- Only the data with date greater than last load date are loaded to the bronze layer

### using sql

In [0]:
%sql
select * from datamodeling.bronze.bronze_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated
1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,129.99,1,129.99,Canada,2026-05-08
1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,89.50,1,89.50,United Kingdom,2026-05-08


In [0]:
if spark.catalog.tableExists('datamodeling.bronze.bronze_table'):
    last_load_date = spark.sql("""
                            select max(order_date) as max_date
                            from datamodeling.bronze.bronze_table
                            """).first()["max_date"]
else:
    last_load_date = '1990-01-01'

In [0]:
last_load_date

datetime.date(2026, 5, 4)

In [0]:
spark.sql(f"""select * from datamodeling.default.source_table
            where order_date > '{last_load_date}'""").createOrReplaceTempView('bronze_source')

In [0]:
%sql
select * from bronze_source

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated
1005,2026-05-05,505,Michael Brown,michael.brown@gmail.com,2005,Wireless Headphones,Electronics,129.99,1,129.99,Canada,2026-05-08
1006,2026-05-06,506,Emma Wilson,emma.wilson@yahoo.com,2006,Coffee Maker,Home Appliances,89.50,1,89.50,United Kingdom,2026-05-08


- Saving the table as CTAS

In [0]:
spark.sql("create schema if not exists datamodeling.bronze")

DataFrame[]

In [0]:
%sql
create or replace table datamodeling.bronze.bronze_table
as
select * from bronze_source

num_affected_rows,num_inserted_rows


In [0]:
%sql
drop table datamodeling.bronze.bronze_table

### Using PySpark

In [0]:
df = spark.sql(f"""select * from datamodeling.default.source_table
                    where order_date > '{last_load_date}'""")

In [0]:
df.display()

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,product_price,quantity,revenue,country,last_updated
1001,2026-05-01,501,John Carter,john.carter@gmail.com,2001,Wireless Mouse,Electronics,25.99,2,51.98,USA,2026-05-07
1002,2026-05-02,502,Emma Watson,emma.watson@yahoo.com,2002,Office Chair,Furniture,149.50,1,149.50,Canada,2026-05-07
1003,2026-05-03,503,Rahul Sharma,rahul.sharma@gmail.com,2003,Mechanical Keyboard,Electronics,89.99,3,269.97,India,2026-05-07
1004,2026-05-04,504,Sophia Lee,sophia.lee@outlook.com,2004,Running Shoes,Sports,79.99,2,159.98,Australia,2026-05-07


In [0]:
df.write.format('delta').mode('append').saveAsTable('datamodeling.bronze.bronze_table')